# exp_20260518_037_realmlp_shared001v2_pitstop_light_min

Base: `exp_20260514_029_realmlp_shared001v2_min`.

Change: remove PitStop-derived categorical/count features only: `PitStop_cat_` and `_PitStop_cat_count`.

Raw `PitStop` is kept. Role: 029 PitStop-light variant.


In [1]:
# ============================================================
# PS S6E5 - exp_20260518_037_realmlp_shared001v2_pitstop_light_min
#
# Base:
#   029: exp_20260518_037_realmlp_shared001v2_pitstop_light_min
#
# Change from 029:
#   Remove PitStop-derived categorical/count features only:
#     - PitStop_cat_
#     - _PitStop_cat_count
#   Keep raw PitStop.
#
# Purpose:
#   Create a 029 PitStop-light variant that weakens PitStop-derived amplification
#   while preserving most of 029's strength.
#
# Notes:
#   - This is not a no-PitStop model.
#   - Raw PitStop remains used.
#   - Race_Year_TE remains unchanged from 029.
#   - Normalized_TyreLife is dropped from original.
#   - OOF/pred .npy, submission, cv_result.json, and memo_draft.yml are saved.
# ============================================================


In [2]:
import os
import sys
import gc
import json
import random
import warnings
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)

In [3]:
# ============================================================
# Install / imports
# ============================================================

try:
    import pytabkit  # noqa: F401
except Exception:
    print("pytabkit not found. Installing pytabkit...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pytabkit"])

import torch
from importlib.metadata import version
from colorama import Fore, Style
from pytabkit import RealMLP_TD_Classifier

pytabkit not found. Installing pytabkit...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 6.3 MB/s eta 0:00:00


In [4]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260518_037_realmlp_shared001v2_pitstop_light_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    TEST_PATH = COMP_BASE / "test.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    ORIG_PATH = Path(
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv"
    )

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID = "id"
    TARGET = "PitNextLap"
    DANGER_COL = "Normalized_TyreLife"

    FOLDS = 5
    SEED = 42
    TE = True

    # 037: remove PitStop-derived categorical/count features only.
    # Raw PitStop remains used.
    DROP_PITSTOP_DERIVED_FEATURES_037 = True
    PITSTOP_DERIVED_DROP_COLS_037 = ["PitStop_cat_", "_PitStop_cat_count"]

    PARAMS = {
        "random_state": 42,
        "verbosity": 2,
        "val_metric_name": "1-auc_ovr",

        "n_ens": 24,
        "n_epochs": 6,
        "batch_size": 256,
        "use_early_stopping": False,
        "early_stopping_additive_patience": 10,
        "early_stopping_multiplicative_patience": 1,

        "lr": 0.01,
        "wd": 0.016,
        "sq_mom": 0.99,
        "lr_sched": "lin_cos_log_15",
        "first_layer_lr_factor": 0.25,

        "embedding_size": 6,
        "max_one_hot_cat_size": 18,
        "hidden_sizes": [512, 256, 128],
        "act": "silu",
        "p_drop": 0.05,
        "p_drop_sched": "invsqrtp1e-3",

        "plr_hidden_1": 16,
        "plr_hidden_2": 8,
        "plr_act_name": "gelu",
        "plr_lr_factor": 0.1151,
        "plr_sigma": 2.33,

        "ls_eps": 0.01,
        "ls_eps_sched": "sqrt_cos",

        "add_front_scale": False,
        "bias_init_mode": "neg-uniform-dynamic-2",
        "tfms": [
            "one_hot",
            "median_center",
            "robust_scale",
            "smooth_clip",
            "embedding",
            "l2_normalize",
        ],
    }


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [5]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def print_section(title: str) -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def per_year_auc(df: pd.DataFrame, y_true, pred) -> dict:
    out = {}
    years = sorted(pd.Series(df["Year"]).dropna().unique().tolist())
    y_arr = np.asarray(y_true).astype(int)
    p_arr = np.asarray(pred, dtype=float)

    for yr in years:
        mask = df["Year"].values == yr
        if mask.sum() == 0 or len(np.unique(y_arr[mask])) < 2:
            out[str(int(yr))] = None
        else:
            out[str(int(yr))] = float(roc_auc_score(y_arr[mask], p_arr[mask]))

    return out


def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in out.columns:
        dtype = out[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            c_min = out[col].min()
            c_max = out[col].max()
            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                out[col] = out[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                out[col] = out[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                out[col] = out[col].astype(np.int32)
        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")

    return out


seed_everything(CFG.SEED)

print("PyTorch version:", torch.__version__)
print("PyTabKit version:", version("pytabkit"))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

PyTorch version: 2.10.0+cu128
PyTabKit version: 1.7.3
GPU: Tesla T4


In [6]:
# ============================================================
# Load data
# ============================================================

print_section("Load Data")

train_raw = pd.read_csv(CFG.TRAIN_PATH)
test_raw = pd.read_csv(CFG.TEST_PATH)
orig_raw = pd.read_csv(CFG.ORIG_PATH)
sub_template = pd.read_csv(CFG.SUB_PATH)

print("Train shape:", train_raw.shape)
print("Test shape :", test_raw.shape)
print("Orig shape :", orig_raw.shape)

assert CFG.ID in train_raw.columns
assert CFG.ID in test_raw.columns
assert CFG.TARGET in train_raw.columns
assert CFG.TARGET in orig_raw.columns

orig_raw = orig_raw.drop(columns=[CFG.DANGER_COL], errors="ignore")


Load Data
Train shape: (439140, 16)
Test shape : (188165, 15)
Orig shape : (101371, 16)


In [7]:
# ============================================================
# Preprocess features
# ============================================================

print_section("Preprocess Features")

y_orig = orig_raw[CFG.TARGET].copy()
orig = orig_raw.drop(columns=[CFG.TARGET])

X = train_raw.drop(columns=[CFG.ID, CFG.TARGET])
train_id = train_raw[CFG.ID].copy()
y = train_raw[CFG.TARGET].copy()

X_test = test_raw.drop(columns=[CFG.ID])
test_id = test_raw[CFG.ID].copy()

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

category_map = {}

important_combos = [
    ("Race", "Compound"),
    ("Race", "Year"),
]


def feature_engineering(df: pd.DataFrame, fit: bool = False):
    df = df.copy()

    # Arithmetic interactions
    df["_LapNumber_/_RaceProgress"] = (
        df["LapNumber"] / (df["RaceProgress"] + 1e-6)
    ).astype("float32")

    df["_TyreLife_/_LapNumber"] = (
        df["TyreLife"] / df["LapNumber"].clip(lower=1)
    ).astype("float32")

    df["_LapTime (s)_*_Cumulative_Degradation"] = (
        df["LapTime (s)"] * df["Cumulative_Degradation"]
    ).astype("float32")

    df["_LapTime (s)_*_Cumulative_Degradation_abs"] = (
        df["LapTime (s)"] * df["Cumulative_Degradation"].abs()
    ).astype("float32")

    df["_LapTime (s)_/_Cumulative_Degradation_abs"] = (
        df["LapTime (s)"] / (df["Cumulative_Degradation"].abs() + 1e-6)
    ).astype("float32")

    # Categorize numericals
    for col in num_cols + ["_LapNumber_/_RaceProgress", "_TyreLife_/_LapNumber"]:
        cat_name = f"{col}_cat_" if col in num_cols else f"{col[1:]}_cat_"

        floored = np.floor(pd.to_numeric(df[col], errors="coerce")).fillna(-999999)

        if fit:
            codes, uniques = pd.factorize(floored, sort=False)
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = floored.map(code_map).fillna(-1).astype("int32")

        df[cat_name] = pd.Series(codes, index=df.index).astype(str)

    # Count encoding
    for col in cat_cols + ["Year_cat_", "PitStop_cat_"]:
        if col not in df.columns:
            continue

        count_name = f"_{col}_count" if col in cat_cols else f"_{col[:-1]}_count"

        if fit:
            count_map = df[col].value_counts()
            category_map[count_name] = count_map
        else:
            count_map = category_map[count_name]

        df[count_name] = df[col].map(count_map).fillna(0).astype("int32")

    # Discretize numericals
    bin_config = {
        "RaceProgress": [200],
        "LapTime (s)": [7],
    }

    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            strategy = "quantile"
            bin_name = f"{col}_{n_bins}_{strategy}_bin_"

            if fit:
                kb = KBinsDiscretizer(
                    n_bins=n_bins,
                    encode="ordinal",
                    strategy=strategy,
                    subsample=None,
                )
                binned = kb.fit_transform(df[[col]]).ravel().astype("int32")
                category_map[bin_name] = kb
            else:
                kb = category_map[bin_name]
                binned = kb.transform(df[[col]]).ravel().astype("int32")

            df[bin_name] = pd.Series(binned, index=df.index).astype(str)

    # Create interaction categories
    combo_names = []

    for cols in important_combos:
        combo_name = "_".join(cols) + "_"
        combo_names.append(combo_name)

        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + "_" + df[col].astype(str)

        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype("int32")

        df[combo_name] = pd.Series(codes, index=df.index).astype(str)

    new_cat_cols = [col for col in df.columns if col.endswith("_")]
    new_num_cols = [col for col in df.columns if col.startswith("_")]

    return df, new_cat_cols, new_num_cols, combo_names


X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, _, _, _ = feature_engineering(X_test, fit=False)
orig, _, _, _ = feature_engineering(orig, fit=False)

cat_cols += new_cat_cols
num_cols += new_num_cols

# 037: PitStop-light variant.
# Remove PitStop-derived categorical/count amplification, but keep raw PitStop.
DROPPED_FEATURES_037 = []
if CFG.DROP_PITSTOP_DERIVED_FEATURES_037:
    for col in CFG.PITSTOP_DERIVED_DROP_COLS_037:
        if col in X.columns:
            DROPPED_FEATURES_037.append(col)

    for df_name, df in [("X", X), ("X_test", X_test), ("orig", orig)]:
        drop_cols = [c for c in DROPPED_FEATURES_037 if c in df.columns]
        if drop_cols:
            df.drop(columns=drop_cols, inplace=True)

    cat_cols = [c for c in cat_cols if c not in DROPPED_FEATURES_037]
    num_cols = [c for c in num_cols if c not in DROPPED_FEATURES_037]

print("DROPPED_FEATURES_037:", DROPPED_FEATURES_037)

X = reduce_mem_usage(X)
X_test = reduce_mem_usage(X_test)
orig = reduce_mem_usage(orig)

print("X shape     :", X.shape)
print("X_test shape:", X_test.shape)
print("orig shape  :", orig.shape)
print("combo_names :", combo_names)

# Guardrails
assert CFG.DANGER_COL not in X.columns
assert CFG.DANGER_COL not in X_test.columns
assert CFG.DANGER_COL not in orig.columns

# 037 guardrails
assert "PitStop" in X.columns
assert "PitStop" in X_test.columns
assert "PitStop" in orig.columns
assert "PitStop_cat_" not in X.columns
assert "_PitStop_cat_count" not in X.columns
assert "PitStop_cat_" not in X_test.columns
assert "_PitStop_cat_count" not in X_test.columns
assert "PitStop_cat_" not in orig.columns
assert "_PitStop_cat_count" not in orig.columns


Preprocess Features
DROPPED_FEATURES_037: ['PitStop_cat_', '_PitStop_cat_count']
X shape     : (439140, 39)
X_test shape: (188165, 39)
orig shape  : (101371, 39)
combo_names : ['Race_Compound_', 'Race_Year_']


In [8]:
# ============================================================
# Train K-Fold
# ============================================================

print_section("Train K-Fold")

skf = StratifiedKFold(
    n_splits=CFG.FOLDS,
    shuffle=True,
    random_state=CFG.SEED,
)

oof_preds = np.zeros(len(X), dtype=np.float32)
test_preds = np.zeros(len(X_test), dtype=np.float32)

fold_scores = []
feature_columns_final = None

for fold, ((tr_idx, val_idx), (or_tr_idx, _)) in enumerate(
    zip(skf.split(X, y), skf.split(orig, y_orig)),
    1,
):
    print("\n" + "#" * 24)
    print(f"Fold {fold}/{CFG.FOLDS}")
    print("#" * 24)

    X_tr = X.iloc[tr_idx].copy()
    orig_tr = orig.iloc[or_tr_idx].copy()

    X_tr = pd.concat(
        [X_tr, orig_tr],
        axis=0,
        ignore_index=True,
    )

    y_tr = pd.concat(
        [y.iloc[tr_idx], y_orig.iloc[or_tr_idx]],
        axis=0,
        ignore_index=True,
    )

    X_val = X.iloc[val_idx].copy()
    y_val = y.iloc[val_idx].copy()
    X_tst = X_test.copy()

    # Fold-safe Target Encoding
    if CFG.TE:
        te_cols = combo_names

        te = TargetEncoder(
            cv=CFG.FOLDS,
            smooth="auto",
            shuffle=True,
            random_state=CFG.SEED,
        )

        tr_enc = te.fit_transform(X_tr[te_cols], y_tr)
        val_enc = te.transform(X_val[te_cols])
        tst_enc = te.transform(X_tst[te_cols])

        te_names = [f"_{col}TE" for col in te_cols]

        X_tr[te_names] = tr_enc
        X_val[te_names] = val_enc
        X_tst[te_names] = tst_enc

    if fold == 1:
        feature_columns_final = X_tr.columns.tolist()
        assert "PitStop" in feature_columns_final
        assert "PitStop_cat_" not in feature_columns_final
        assert "_PitStop_cat_count" not in feature_columns_final
        print("len(FEATURES):", len(feature_columns_final))
        pd.Series(feature_columns_final).to_csv(
            CFG.OUTDIR / "feature_columns.csv",
            index=False,
            header=False,
        )

    model = RealMLP_TD_Classifier(**CFG.PARAMS)
    model.fit(X_tr, y_tr, X_val, y_val)

    val_preds = model.predict_proba(X_val)[:, 1].astype(np.float32)
    fold_test_preds = model.predict_proba(X_tst)[:, 1].astype(np.float32)

    oof_preds[val_idx] = val_preds
    test_preds += fold_test_preds / CFG.FOLDS

    fold_score = roc_auc_score(y_val, val_preds)
    fold_scores.append(float(fold_score))

    print(f"{Fore.GREEN}{Style.BRIGHT}Fold {fold} | AUC Score: {fold_score:.6f}{Style.RESET_ALL}")

    del X_tr, X_val, X_tst, model, val_preds, fold_test_preds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Train K-Fold

########################
Fold 1/5
########################
len(FEATURES): 41
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_LapTime (s)_*_Cumulative_Degradation', '_LapTime (s)_*_Cumulative_Degradation_abs', '_LapTime (s)_/_Cumulative_Degradation_abs', '_Driver_count', '_Compound_count', '_Race_count', '_Year_cat_count', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'LapNumber_/_RaceProgress_cat_', 'TyreLife_/_LapNumber_cat_', 'RaceProgress_200_quantile_bin_', 'LapTime (s)_7_quantile_bin_', 'Race_Compound_', 'Race_Year_']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/6: val 1-auc_ovr = 0.059051
Epoch 2/6: val 1-auc_ovr = 0.050628
Epoch 3/6: val 1-auc_ovr = 0.048206
Epoch 4/6: val 1-auc_ovr = 0.046510
Epoch 5/6: val 1-auc_ovr = 0.045069
Epoch 6/6: val 1-auc_ovr = 0.044906


`Trainer.fit` stopped: `max_epochs=6` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 1 | AUC Score: 0.955094

########################
Fold 2/5
########################
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_LapTime (s)_*_Cumulative_Degradation', '_LapTime (s)_*_Cumulative_Degradation_abs', '_LapTime (s)_/_Cumulative_Degradation_abs', '_Driver_count', '_Compound_count', '_Race_count', '_Year_cat_count', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'LapNumber_/_RaceProgress_cat_', 'TyreLife_/_LapNumber_cat_', 'RaceProgress_200_quantile_bin_', 'LapTime (s)_7_quantile_bin_', 'Race_Compound_', 'Race_Year_']


...

Fold 4 | AUC Score: 0.953263

########################
Fold 5/5
########################
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_LapTime (s)_*_Cumulative_Degradation', '_LapTime (s)_*_Cumulative_Degradation_abs', '_LapTime (s)_/_Cumulative_Degradation_abs', '_Driver_count', '_Compound_count', '_Race_count', '_Year_cat_count', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'LapNumber_/_RaceProgress_cat_', 'TyreLife_/_LapNumber_cat_', 'RaceProgress_200_quantile_bin_', 'LapTime (s)_7_quantile_bin_', 'Race_Compound_', 'Race_Year_']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/6: val 1-auc_ovr = 0.060105
Epoch 2/6: val 1-auc_ovr = 0.050960
Epoch 3/6: val 1-auc_ovr = 0.048341
Epoch 4/6: val 1-auc_ovr = 0.046678
Epoch 5/6: val 1-auc_ovr = 0.045179
Epoch 6/6: val 1-auc_ovr = 0.045061


`Trainer.fit` stopped: `max_epochs=6` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 5 | AUC Score: 0.954939


In [9]:
# ============================================================
# Evaluation and Save Artifacts
# ============================================================

print_section("Evaluation and Save Artifacts")

oof_score = roc_auc_score(y, oof_preds)
year_auc = per_year_auc(train_raw, y.values, oof_preds)

print(f"Overall OOF AUC: {oof_score:.9f}")
print("fold_scores:", fold_scores)
print("per_year_auc:", year_auc)

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof_preds.astype(np.float32))
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", test_preds.astype(np.float32))

oof_df = pd.DataFrame({
    CFG.ID: train_id.values,
    CFG.TARGET: oof_preds,
})
oof_df.to_csv(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv", index=False)

sub_out = sub_template.copy()
target_col = [c for c in sub_out.columns if c != CFG.ID][0]
sub_out[target_col] = np.clip(test_preds, 1e-6, 1 - 1e-6)

submission_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub_out.to_csv(submission_path, index=False)

feature_detail = pd.DataFrame({
    "feature": feature_columns_final,
})
feature_detail["is_te"] = feature_detail["feature"].isin([f"_{c}TE" for c in combo_names])
feature_detail["is_dropped_pitstop_derived_037"] = feature_detail["feature"].isin(CFG.PITSTOP_DERIVED_DROP_COLS_037)
feature_detail["contains_pitstop"] = feature_detail["feature"].str.contains("PitStop", regex=False)
feature_detail["is_combo"] = feature_detail["feature"].isin(combo_names)
feature_detail["is_cat"] = feature_detail["feature"].str.endswith("_")
feature_detail["is_count"] = feature_detail["feature"].str.endswith("_count")
feature_detail["is_pitstop_related"] = feature_detail["feature"].str.contains("PitStop", regex=False)
feature_detail.to_csv(CFG.OUTDIR / "feature_columns_detail.csv", index=False)

cv_result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
        "created_at": "2026-05-18",
    },
    "source": {
        "reference_code": "ps-s6e5-029-realmlp-shared001v2-min.ipynb / code_shared_001v2.txt",
        "family": "RealMLP",
        "description": "029 shared001v2 RealMLP PitStop-light variant: remove PitStop_cat_ and _PitStop_cat_count only.",
        "base_experiment": "exp_20260514_029_realmlp_shared001v2_min",
    },
    "data": {
        "train_shape": list(train_raw.shape),
        "test_shape": list(test_raw.shape),
        "orig_shape_after_drop": list(orig_raw.drop(columns=[CFG.DANGER_COL], errors="ignore").shape),
        "target_rate_train": float(train_raw[CFG.TARGET].mean()),
        "target_rate_orig": float(y_orig.mean()),
        "danger_col_used": False,
    },
    "features": {
        "feature_count": len(feature_columns_final),
        "combo_names": combo_names,
        "te_names": [f"_{c}TE" for c in combo_names],
        "dropped_pitstop_derived_features_037": DROPPED_FEATURES_037,
        "raw_pitstop_kept_037": "PitStop" in feature_columns_final,
        "pitstop_used": bool(any("PitStop" in c for c in feature_columns_final)),
    },
    "model": {
        "family": "RealMLP",
        "estimator": "RealMLP_TD_Classifier",
        "folds": CFG.FOLDS,
        "seed": CFG.SEED,
        "params": CFG.PARAMS,
    },
    "result": {
        "cv_auc": float(oof_score),
        "public_lb": None,
        "fold_scores": [float(x) for x in fold_scores],
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
        "per_year_auc": year_auc,
    },
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "oof_npy": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred_npy": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "oof_csv": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv"),
        "submission": str(submission_path),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "feature_columns_detail": str(CFG.OUTDIR / "feature_columns_detail.csv"),
        "cv_result": str(CFG.OUTDIR / "cv_result.json"),
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(cv_result, f, ensure_ascii=False, indent=2)

memo_yml = f"""experiment:
  id: exp_20260518_037_realmlp_shared001v2_pitstop_light_min
  title: RealMLP shared001v2 PitStop-light
  competition: PS S6E5 Predicting F1 Pit Stops
  metric: AUC
  status: completed

objective:
  summary: >
    029 RealMLP shared001v2をbaseとして、PitStop由来の派生特徴だけを弱める。
    具体的には PitStop_cat_ と _PitStop_cat_count を削除し、raw PitStopは残す。
    029の強さをある程度維持しながら、029/036とは違う誤差を作れるか確認する。
  base_experiment: exp_20260514_029_realmlp_shared001v2_min
  intended_role: 029_pitstop_light_variant

source:
  reference_code: ps-s6e5-029-realmlp-shared001v2-min.ipynb
  model_family: RealMLP
  notes:
    - RealMLP_TD_Classifier
    - 029 variant
    - remove PitStop_cat_ and _PitStop_cat_count only
    - keep raw PitStop
    - keep Race_Year_TE unchanged from 029
    - Normalized_TyreLife is dropped if present

data:
  competition_train: /kaggle/input/competitions/playground-series-s6e5/train.csv
  competition_test: /kaggle/input/competitions/playground-series-s6e5/test.csv
  original: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv
  target: PitNextLap
  caution:
    - raw PitStop remains used
    - PitStop_cat_ and _PitStop_cat_count are removed
    - Normalized_TyreLife is dropped
    - This is not a no-PitStop model

features:
  combo_features: {combo_names}
  target_encoding_columns: {combo_names}
  target_encoding_names: {[f"_{c}TE" for c in combo_names]}
  dropped_pitstop_derived_features_037: {DROPPED_FEATURES_037}
  guardrails:
    raw_pitstop_kept: {"PitStop" in feature_columns_final}
    pitstop_cat_removed: {"PitStop_cat_" not in feature_columns_final}
    pitstop_cat_count_removed: {"_PitStop_cat_count" not in feature_columns_final}
  notable_features:
    - PitStop
    - Race_Year_
    - _Race_Year_TE
    - RaceProgress_200_quantile_bin_
    - LapTime (s)_7_quantile_bin_
    - floored_numerical_categories
    - count_encoding

model:
  family: RealMLP
  estimator: RealMLP_TD_Classifier
  folds: 5
  seed: 42
  params:
    n_ens: 24
    n_epochs: 6
    batch_size: 256
    lr: 0.01
    wd: 0.016
    hidden_sizes:
      - 512
      - 256
      - 128
    ls_eps: 0.01

result:
  cv_auc: {float(oof_score)}
  public_lb: null
  fold_scores: {[float(x) for x in fold_scores]}
  fold_mean: {float(np.mean(fold_scores))}
  fold_std: {float(np.std(fold_scores))}
  per_year_auc: {year_auc}
  compared_to_029:
    cv_auc: 0.9540420784720796
    public_lb: 0.95397
    note: >
      037 is expected to lose some CV if PitStop-derived features are important.
      Judge primarily by corr_vs_029/036 and blend weight.
  compared_to_036:
    cv_auc: 0.9540748794531666
    public_lb: 0.95393
    note: >
      036 is the successful no_Race_Year_TE variant and current strong 029-family candidate.

artifacts:
  notebook: ps-s6e5-037-realmlp-shared001v2-pitstop-light-min.ipynb
  oof: oof_exp_20260518_037_realmlp_shared001v2_pitstop_light_min.npy
  pred: pred_exp_20260518_037_realmlp_shared001v2_pitstop_light_min.npy
  submission: submission_exp_20260518_037_realmlp_shared001v2_pitstop_light_min.csv
  cv_result: cv_result.json
  feature_columns: feature_columns.csv
  feature_columns_detail: feature_columns_detail.csv

blend_policy:
  benchmark_code: 033_based_simplified
  add_to_stack:
    - "014"
    - "015"
    - "016"
    - "017"
    - "021"
    - "026"
    - "028"
    - "029"
    - "032"
    - "036"
    - "037"
  decision_focus:
    - single_cv_vs_029
    - single_cv_vs_036
    - corr_vs_029
    - corr_vs_036
    - corr_vs_032
    - nm_hc_slsqp_weight
    - whether_037_gets_weight_alongside_029_036
    - public_lb_if_submitted
  keep_rule: >
    Keep if 037 maintains reasonable single strength and receives meaningful blend weight,
    especially if it complements 036 rather than simply duplicating 029.
  drop_rule: >
    Drop if 037 is simply a weaker high-correlation copy of 029/036 and receives near-zero blend weight.

judgement:
  status: pending
  summary: >
    CV/LB/correlation/blend weight確認後に keep / hold / drop を判断する。
"""
with open(CFG.OUTDIR / "memo_draft.yml", "w", encoding="utf-8") as f:
    f.write(memo_yml)

print("Saved standardized artifacts to:", CFG.OUTDIR)
print("Submission:", submission_path)
display(sub_out.head())


Evaluation and Save Artifacts
Overall OOF AUC: 0.954012631
fold_scores: [0.9550936306855496, 0.9530757581123764, 0.9540173557427704, 0.9532632244263591, 0.9549389130558753]
per_year_auc: {'2022': 0.9194077746323744, '2023': 0.9490754496590372, '2024': 0.934209723346015, '2025': 0.9341707893250599}
Saved standardized artifacts to: /kaggle/working/exp_20260518_037_realmlp_shared001v2_pitstop_light_min
Submission: /kaggle/working/exp_20260518_037_realmlp_shared001v2_pitstop_light_min/submission_exp_20260518_037_realmlp_shared001v2_pitstop_light_min.csv


,id,PitNextLap
0,439140,0.003961
1,439141,0.019869
2,439142,0.006358
3,439143,0.216826
4,439144,0.794868
